# Учебный блокнот для главы 17
**Создайте свою первую LLM — Глава 17: Варианты развертывания**

Этот блокнот объединяет исполняемые примеры кода из главы 17. Развертывание Modal выполняется из терминала, а не из Colab, но этот блокнот объясняет каждый шаг.

- Установки: modal
- Данные: встроенные примеры; внешние файлы не требуются
- Среда выполнения: Большинство команд Modal выполняются в терминале, а не в блокноте

In [ ]:
# ===== НАСТРОЙКА =====
# Установим Modal
!pip install -q modal

print('Modal установлен!')
print('ВАЖНО: Выполните "modal setup" в терминале для аутентификации.')
print('Это откроет окно браузера для входа.')

## Раздел 17.1: Зачем нужен Modal?

**Что такое бессерверные вычисления (Serverless)?**

| Традиционный сервер | Бессерверные (Modal) |
|---------------------|----------------------|
| Вы арендуете компьютер 24/7 | Код выполняется только при необходимости |
| Вы платите даже когда он простаивает | Вы платите за секунды вычислений |
| Вы управляете обновлениями, безопасностью | Платформа управляет всем |
| Фиксированная мощность | Автоматическое масштабирование |

**Аналогия:** Традиционный хостинг похож на владение автомобилем. Бессерверные вычисления (Serverless) похожи на использование такси, вы платите только когда едете.

**Зачем использовать Modal для LLM?**
- Встроенная поддержка GPU (от T4 до H100)
- $30/месяц бесплатных кредитов (~37 часов GPU L4; ~50 часов на T4)
- Встроенная поддержка Python (без Docker, без YAML)
- Масштабирование до нуля (нет платы за простой)

## Раздел 17.2: Привет, Modal

Простейшее приложение Modal. Сохраните это как `hello.py` и запустите с помощью `modal run hello.py`:

In [ ]:
# Сохраните это как hello.py
hello_modal_code = '''
import modal

app = modal.App("hello-world")

@app.function()
def hello(name: str) -> str:
    return f"Hello, {name}!"

@app.local_entrypoint()
def main():
    result = hello.remote("World")
    print(result)
'''

print("Сохраните этот код как 'hello.py':")
print(hello_modal_code)
print("\nЗатем выполните: modal run hello.py")

**Что только что произошло?**

1. `modal.App()` создает ваше приложение
2. `@app.function()` помечает код для выполнения в облаке Modal
3. `hello.remote()` вызывает функцию удаленно
4. Modal запустил контейнер, выполнил ваш код, вернул результат

## Раздел 17.3: Добавление поддержки GPU

Один параметр добавляет поддержку GPU:

In [ ]:
# Сохраните это как gpu_test.py
gpu_test_code = '''
import modal

app = modal.App("gpu-test")

# torch должен быть установлен в образ контейнера (следующий раздел — об образах)
image = modal.Image.debian_slim(python_version="3.11").pip_install("torch")

@app.function(image=image, gpu="L4")  # Вот и всё!
def check_gpu():
    import torch
    if torch.cuda.is_available():
        device = torch.cuda.get_device_name(0)
        return f"GPU available: {device}"
    return "No GPU found"

@app.local_entrypoint()
def main():
    print(check_gpu.remote())
'''

print("Сохраните этот код как 'gpu_test.py':")
print(gpu_test_code)
print("\nЗатем выполните: modal run gpu_test.py")
print("\nОжидаемый вывод: GPU available: NVIDIA L4")

**Один параметр.** Без установки CUDA, без управления драйверами, без образов Docker. Modal управляет всем стеком GPU.

## Раздел 17.4: Установка зависимостей

Вашей LLM нужны библиотеки. Modal позволяет определить окружение контейнера на Python:

In [ ]:
# Сохраните это как with_deps.py
with_deps_code = '''
import modal

# Определяем образ контейнера
image = modal.Image.debian_slim(python_version="3.11").pip_install(
    "torch",
    "transformers",
    "accelerate",
)

app = modal.App("with-deps")

@app.function(image=image, gpu="T4")
def generate_text():
    from transformers import pipeline
    
    generator = pipeline("text-generation", model="gpt2", device=0)
    result = generator("The meaning of life is", max_length=50)
    return result[0]["generated_text"]

@app.local_entrypoint()
def main():
    print(generate_text.remote())
'''

print("Сохраните этот код как 'with_deps.py':")
print(with_deps_code)
print("\nЗатем выполните: modal run with_deps.py")

Параметр `image` сообщает Modal, что нужно установить. Первый запуск занимает больше времени (построение образа), но последующие запуски используют кешированный образ.

## Раздел 17.5: Веб-конечные точки с FastAPI

Создайте веб-конечную точку, к которой может получить доступ любой. Здесь пригодятся знания FastAPI из главы 16:

In [ ]:
# Сохраните это как simple_api.py
simple_api_code = '''
import modal
from fastapi import FastAPI

app = modal.App("my-api")
web_app = FastAPI()

# Зависимости нужны в образе контейнера, а не только на вашем компьютере.
image = modal.Image.debian_slim(python_version="3.11").pip_install(
    "fastapi==0.163.2",
    "uvicorn==0.54.0",
    "httpx==0.28.1",
    "pydantic==2.14.3",
)

@web_app.get("/health")
def health():
    return {"status": "healthy"}

@web_app.get("/hello/{name}")
def hello(name: str):
    return {"message": f"Hello, {name}!"}

@app.function(image=image)
@modal.asgi_app()
def serve():
    return web_app
'''

print("Сохраните этот код как 'simple_api.py':")
print(simple_api_code)
print("\nЗатем выполните: modal deploy simple_api.py")
print("\nВы получите URL вида: https://your-workspace--my-api-serve.modal.run")

**Ваш API запущен!** Откройте этот URL в браузере. Добавьте `/health` к пути:

```
https://your-workspace--my-api-serve.modal.run/health
```

Вы должны увидеть: `{"status": "healthy"}`

## Раздел 17.6: Полноценный сервис LLM

Вот готовое к работе развертывание LLM:

Прокси Modal проверяет пару `Modal-Key`/`Modal-Secret` до запуска платного контейнера. После этого `/chat` отдельно проверяет bearer-токен приложения.

In [ ]:
# Сохраните это как llm_service.py
# (Тот же сервис, что в книге: защитные ограничения главы 16 + subprocess vllm serve.
#  Проверено по документации vLLM/Modal 2026-11; для развертывания нужны аккаунт Modal + GPU.)
llm_service_code = '''
import modal
from fastapi import FastAPI, HTTPException, Request, Depends
from fastapi.security import HTTPBearer, HTTPAuthorizationCredentials
from fastapi.responses import JSONResponse
from pydantic import BaseModel, field_validator
import os
from collections import defaultdict
import threading
import uuid
from datetime import datetime, timedelta
import subprocess
import time
import requests

MODEL = "Qwen/Qwen3.5-2B"
# Note: no trust_remote_code needed. Current models like Qwen3.5 are
# natively supported by vLLM; only enable it for repos whose custom
# code you have reviewed, since it executes code from the model repo.

image = modal.Image.debian_slim(python_version="3.11").pip_install(
    "fastapi",
    "vllm==0.34.0",  # Pin for reproducibility (current stable, checked 2026-11)
    "torch",
    "transformers",
    "requests",
)

app = modal.App("my-llm-service")
model_cache = modal.Volume.from_name("model-cache", create_if_missing=True)
web_app = FastAPI(title="My LLM API", version="1.0.0")

# Declaring the bearer scheme gives Swagger's "Authorize" button and marks
# protected routes in the docs; require_token below does the actual check.
bearer_scheme = HTTPBearer(auto_error=False)

# ===== Chapter 16 guardrails, plus auth and token budgets =====
MAX_PROMPT_CHARS = 4000  # cheap first filter; the real budget is in tokens
API_TOKEN = os.environ.get("API_TOKEN", "")  # set via Modal secret, see below
MAX_INPUT_TOKENS = 2048  # templated input budget (with the output cap, fits 8192)
MAX_OUTPUT_TOKENS = 512  # server-side cap: no client can bill you for more
RATE_LIMIT = 20
RATE_WINDOW = timedelta(minutes=1)
request_counts: dict[str, list[datetime]] = defaultdict(list)
_rate_lock = threading.Lock()

def require_token(credentials: HTTPAuthorizationCredentials | None = Depends(bearer_scheme)):
    """Аутентификация Bearer-токеном: общий GPU-эндпоинт не должен быть открыт всему миру."""
    if not API_TOKEN or credentials is None or credentials.credentials != API_TOKEN:
        raise HTTPException(401, "Missing or invalid token")

class ChatRequest(BaseModel):
    message: str
    max_tokens: int = 256

    @field_validator("message")
    @classmethod
    def not_empty(cls, v):
        if not v.strip():
            raise ValueError("Message cannot be empty")
        if len(v) > MAX_PROMPT_CHARS:
            raise ValueError(f"Message too long (max {MAX_PROMPT_CHARS} chars)")
        return v

    @field_validator("max_tokens")
    @classmethod
    def cap_output(cls, v):
        if v < 1:
            raise ValueError("max_tokens must be positive")
        # Clamp instead of reject: clients asking for more get the cap, not an error
        return min(v, MAX_OUTPUT_TOKENS)

class ChatResponse(BaseModel):
    response: str
    latency_ms: float

def check_rate_limit(request: Request):
    client_ip = request.client.host
    now = datetime.now()
    # Check-and-update under one lock: these endpoints run in FastAPI's
    # threadpool, so without it two concurrent requests could both read
    # len == RATE_LIMIT - 1 and both slip through.
    with _rate_lock:
        recent = [t for t in request_counts[client_ip] if now - t < RATE_WINDOW]
        if len(recent) >= RATE_LIMIT:
            request_counts[client_ip] = recent
            raise HTTPException(429, "Too many requests. Please wait a minute.")
        recent.append(now)
        request_counts[client_ip] = recent

def log_request(request_id, input_tokens, output_tokens, latency_ms, status):
    # Allowlisted metadata only: never log prompts, responses, or raw
    # exception text. Modal captures stdout.
    import json
    print(json.dumps({"ts": datetime.now().isoformat(),
                      "request_id": request_id,
                      "input_tokens": input_tokens,
                      "output_tokens": output_tokens,
                      "latency_ms": round(latency_ms, 2),
                      "status": status}))

# ===== vLLM server management (one per container) =====
_server = None
_tokenizer = None

def start_vllm():
    """Start vllm serve once per container and wait until it is ready."""
    global _server, _tokenizer
    if _server is None:
        _server = subprocess.Popen([
            "vllm", "serve", MODEL,
            "--port", "8000",
            "--max-model-len", "8192",       # cap context: keep KV cache small on L4's 24GB
            "--gpu-memory-utilization", "0.90",
        ])
        # Wait for the server's own cheap /health endpoint
        for _ in range(120):
            try:
                requests.get("http://localhost:8000/health", timeout=2).raise_for_status()
                break
            except Exception:
                time.sleep(2)
        else:
            raise RuntimeError("vLLM server did not become ready")
        from transformers import AutoTokenizer
        _tokenizer = AutoTokenizer.from_pretrained(MODEL)
    return _tokenizer

@web_app.get("/health")
def health():
    """Cheap probe: ask the local vLLM server, no generation."""
    try:
        requests.get("http://localhost:8000/health", timeout=2).raise_for_status()
        return {"status": "healthy", "model": "Qwen3.5-2B"}
    except Exception:
        # 503, not 200: monitoring should see a degraded service as down
        return JSONResponse(status_code=503,
                            content={"status": "degraded", "model": "Qwen3.5-2B"})

@web_app.post("/chat", response_model=ChatResponse)
def chat(request: ChatRequest, http_request: Request,
         _auth: None = Depends(require_token)):
    """Generate a response: template the prompt, let vLLM do the inference."""
    check_rate_limit(http_request)
    request_id = uuid.uuid4().hex[:12]
    start = time.time()
    try:
        tokenizer = start_vllm()
        # Instruct-модели ожидают свой шаблон чата, а не сырой текст.
        # apply_chat_template оборачивает сообщение в специальные токены модели.
        messages = [{"role": "user", "content": request.message}]
        # Real token budget: count the templated input with the same
        # tokenizer vLLM uses. Characters are not tokens. With one
        # conversation, apply_chat_template returns a dict-like batch
        # whose "input_ids" is the flat token list; len() of the batch
        # itself would count keys (2), not tokens.
        input_ids = tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True,
            return_dict=True)["input_ids"]
        if len(input_ids) > MAX_INPUT_TOKENS:
            raise HTTPException(
                422, f"Message too long ({len(input_ids)} tokens; "
                     f"max {MAX_INPUT_TOKENS} input tokens)")
        # Forward the SAME ids we just counted: /v1/completions accepts
        # a token-id list as "prompt", so vLLM cannot re-tokenize the
        # text into something different from what we budgeted.
        r = requests.post(
            "http://localhost:8000/v1/completions",
            json={"model": MODEL, "prompt": input_ids,
                  "max_tokens": request.max_tokens, "temperature": 0.7},
            timeout=120,
        )
        r.raise_for_status()
        body = r.json()
        text = body["choices"][0]["text"].strip()
        latency_ms = (time.time() - start) * 1000
        log_request(request_id, len(input_ids),
                    body.get("usage", {}).get("completion_tokens", 0),
                    latency_ms, 200)
        return ChatResponse(response=text, latency_ms=round(latency_ms, 2))
    except HTTPException as e:
        log_request(request_id, 0, 0, (time.time() - start) * 1000, e.status_code)
        raise
    except requests.HTTPError as e:
        # vLLM itself rejected the request (e.g. context overflow): pass
        # 4xx through with a sanitized message, never the backend's text.
        status = e.response.status_code if e.response is not None else 502
        if 400 <= status < 500:
            log_request(request_id, 0, 0, (time.time() - start) * 1000, status)
            raise HTTPException(status, "The model rejected this request. "
                                        "Try a shorter message.")
        log_request(request_id, 0, 0, (time.time() - start) * 1000, 502)
        raise HTTPException(502, "Inference backend error. Please try again.")
    except Exception:
        log_request(request_id, 0, 0, (time.time() - start) * 1000, 500)
        raise HTTPException(500, "Generation failed. Please try again.")

@app.function(
    image=image,
    gpu="L4",
    timeout=600,
    scaledown_window=300,  # Keep warm for 5 minutes
    volumes={"/root/.cache/huggingface": model_cache},  # model downloads once
    secrets=[modal.Secret.from_name("llm-api-token")],  # provides API_TOKEN
)
@modal.concurrent(max_inputs=10)  # Safe: the vLLM server batches concurrent requests
@modal.asgi_app(requires_proxy_auth=True)
def serve():
    start_vllm()  # Load once when the container starts, reuse for every request
    return web_app

'''

print("Сохраните этот код как 'llm_service.py':")
print(llm_service_code)

In [ ]:
print("Сначала создайте секрет аутентификации:")
print("modal secret create llm-api-token API_TOKEN=ваш-длинный-случайный-токен")
print("\nРазверните с помощью: modal deploy llm_service.py")
print("\nПервое развертывание занимает несколько минут (загрузка модели).")
print("Последующие развертывания быстрые.")
print("\nСоздайте прокси-токен Modal и задайте MODAL_KEY и MODAL_SECRET в окружении клиента.")
print('curl -i -H "Modal-Key: $MODAL_KEY" -H "Modal-Secret: $MODAL_SECRET" https://your-url.modal.run/health   # ожидайте 200 healthy')
print('curl -i -H "Modal-Key: $MODAL_KEY" -H "Modal-Secret: $MODAL_SECRET" -X POST https://your-url.modal.run/chat -H "Authorization: Bearer ваш-длинный-случайный-токен" -H "Content-Type: application/json" -d \'{"message": "What is Python?"}\'   # ожидайте 200')
print('curl -i -H "Modal-Key: $MODAL_KEY" -H "Modal-Secret: $MODAL_SECRET" -X POST https://your-url.modal.run/chat -H "Authorization: Bearer wrong-token" -H "Content-Type: application/json" -d \'{"message": "hi"}\'   # ожидайте 401')
print("\nПо завершении остановите: modal app stop my-llm-service")


### Контрольный список первого запуска

После развёртывания выполните эти проверки сами (мы не запускали их на живом развёртывании; каждый ожидаемый результат — то, что вам нужно подтвердить):

```bash
# 0. Создайте секрет аутентификации ДО развёртывания
modal secret create llm-api-token API_TOKEN=ваш-длинный-случайный-токен

# Задайте MODAL_KEY и MODAL_SECRET из прокси-токена в окружении клиента.
# 1. Проверка здоровья: ожидайте HTTP 200 и {"status": "healthy", ...}
curl -i -H "Modal-Key: $MODAL_KEY" -H "Modal-Secret: $MODAL_SECRET" https://your-url.modal.run/health

# 2. Чат с вашим токеном: ожидайте HTTP 200 и JSON-ответ
curl -i -H "Modal-Key: $MODAL_KEY" -H "Modal-Secret: $MODAL_SECRET" -X POST https://your-url.modal.run/chat \
  -H "Authorization: Bearer ваш-длинный-случайный-токен" \
  -H "Content-Type: application/json" \
  -d '{"message": "What is Python?"}'

# 3. Чат с неверным токеном: ожидайте HTTP 401
curl -i -H "Modal-Key: $MODAL_KEY" -H "Modal-Secret: $MODAL_SECRET" -X POST https://your-url.modal.run/chat \
  -H "Authorization: Bearer wrong-token" \
  -H "Content-Type: application/json" \
  -d '{"message": "What is Python?"}'

# 4. Завершите работу после экспериментов (останавливает всю тарификацию GPU)
modal app stop my-llm-service
```

Если шаг 1 возвращает 503 сразу после холодного старта, дайте контейнеру минуту на загрузку модели; постоянное 503 означает, что vLLM не запустился — проверьте `modal app logs my-llm-service`. Обычная вкладка браузера не передаёт прокси-заголовки: используйте curl или другой HTTP-клиент.


## Раздел 17.7: Секреты и тома

### Создание секретов

```bash
# Создаем секрет из командной строки
modal secret create my-secrets API_KEY=your_secret_key
```

### Использование секретов в коде

In [ ]:
# Использование секретов в вашем коде
secrets_example = '''
@app.function(secrets=[modal.Secret.from_name("my-secrets")])
def with_secrets():
    import os
    api_key = os.environ["API_KEY"]
    # Безопасно используем ключ...
'''

print("Использование секретов:")
print(secrets_example)

In [ ]:
# Использование томов для кеширования моделей
volumes_example = '''
# Создаем том для кеша моделей
model_cache = modal.Volume.from_name("model-cache", create_if_missing=True)

@app.function(
    gpu="L4",
    volumes={"/root/.cache/huggingface": model_cache},
)
def with_cache():
    # Модели загружаются один раз, затем кешируются в томе
    pass
'''

print("Использование томов для кеширования моделей:")
print(volumes_example)
print("\nПервый запрос загружает модель. Последующие запросы используют кеш.")

## Раздел 17.8: Варианты GPU

| Размер модели | Рекомендуемый GPU | Стоимость/час |
|---------------|-------------------|---------------|
| < 3 млрд параметров (FP16/квантованные) | T4 (16GB) | $0.59 |
| < 3 млрд параметров (BF16, напр. Qwen3.5-2B) | L4 (24GB) | $0.80 |
| 3-8 млрд параметров | A10 (24GB) | $1.10 |
| 8-30 млрд параметров | A100-40GB | $2.50 |
| 30+ млрд параметров | A100-80GB / H100 | $4-8 |

(Цены проверены 2026-11; уточняйте на modal.com/pricing.)

**Начните с L4.** Веса Qwen3.5-2B хранятся в BF16, а vLLM 0.34 отказывается запускать BF16 на GPU старше Ampere (compute capability 8.0): T4 — это 7.5, и запуск завершился бы ошибкой, а L4 (8.9) и A10 (8.6) проходят порог. T4 по-прежнему подходит для небольших FP16- или квантованных моделей. Обновляйте только при необходимости.

In [ ]:
# Примеры выбора GPU
print("Выбор GPU это один параметр:")
print()
print('@app.function(gpu="T4")      # 16GB, $0.59/час - только FP16 или квантованные модели (без BF16)')
print('@app.function(gpu="L4")      # 24GB, $0.80/час - наш выбор для Qwen3.5-2B')
print('@app.function(gpu="A10")     # 24GB, $1.10/час - та же память, быстрее вычисления')
print('@app.function(gpu="A100")    # Высокая производительность')
print('@app.function(gpu="H100")    # Максимальная мощность')
print('@app.function(gpu="A100:2")  # Два A100')

## Раздел 17.9: Управление развертыванием

### Просмотр логов

```bash
modal app logs my-llm-service
```

Или используйте панель управления на [modal.com](https://modal.com).

### Обновление приложения

```bash
# Просто повторно разверните
modal deploy llm_service.py
```

Без простоев. Modal автоматически обрабатывает непрерывные обновления.

### Контроль затрат

- Используйте `scaledown_window` для выключения простаивающих контейнеров
- Начните с L4, обновляйте только при необходимости
- Настройте оповещения о расходах в панели Modal; `modal app stop my-llm-service` полностью останавливает сервис
- Отслеживайте расходы на панели управления Modal

## Резюме

Вы научились:

1. **Развертывать функции** в Modal с помощью `@app.function()`
2. **Добавлять поддержку GPU** с помощью одного параметра: `gpu="L4"`
3. **Устанавливать зависимости** используя `modal.Image`
4. **Создавать веб-конечные точки** с помощью FastAPI + `@modal.asgi_app()`
5. **Безопасно хранить секреты** с помощью `modal.Secret`
6. **Кешировать модели** с помощью `modal.Volume`
7. **Контролировать затраты** с помощью таймаутов простоя и мониторинга

**Ваша LLM больше не привязана к вашему ноутбуку.** Она доступна авторизованным клиентам, с ускорением GPU.

**Команды для запоминания:**
```bash
pip install modal      # Установка
modal setup            # Аутентификация
modal run file.py      # Запустить один раз
modal deploy file.py   # Развернуть постоянно
modal app logs name    # Просмотр логов
```